
To install, ´conda env create -f environment-HGQ.yml´, or to update env ´conda env update -f environment-HGQ.yml´. Remember to restart kernel.

In [20]:
import os
model_to_test = 'hgq2'
model_revision = 1
hls4ml_revision = 'Vitis_latency_reusefactor4'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, str(model_revision))
os.makedirs(model_dir, exist_ok=True)

description = """
# Model Configuration

Testing HGQ2 with a base model from Sergei.
HLS4ML-config: 'strategy = latency' og 'reusefactor = 4'


- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)



In [22]:
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH'] # 2025 er allerede i path

In [23]:
# Use absolute paths for data files
x_train_val_path = os.path.join(base_dir, "x_train_val.npy")
x_test_path = os.path.join(base_dir, "x_test.npy")
y_train_val_path = os.path.join(base_dir, "y_train_val.npy")
y_test_path = os.path.join(base_dir, "y_test.npy")
classes_path = os.path.join(base_dir, "classes.npy")

x_train_val = np.load(x_train_val_path)
x_test = np.load(x_test_path)
y_train_val = np.load(y_train_val_path)
y_test = np.load(y_test_path)



In [24]:
# Convert dataset arrays to float32
x_train_val = x_train_val.astype(np.float32)
x_test = x_test.astype(np.float32)
y_train_val = y_train_val.astype(np.float32)
y_test = y_test.astype(np.float32)

Load existing model, or create and train a new

In [25]:
keras_model_path = os.path.join(model_dir, f"model_HGQ.keras")

import hgq.layers
from keras.models import load_model
model = load_model(keras_model_path)


/home/ncgadmin/miniconda3/envs/devenv-hgq/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 31 variables whereas the saved optimizer has 60 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:
model.predict(x_test[:10])

2026-03-18 11:53:30.982949: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 424ms/step                                                            


array([[ 2.5 ,  1.  , -1.5 ,  0.  ,  0.  ],
       [-1.  ,  1.75, -3.5 ,  0.  ,  0.  ],
       [-0.5 , -0.25,  1.  , -2.5 , -5.  ],
       [-1.  , -1.  ,  4.  , -3.  , -6.  ],
       [ 0.  , -0.75,  4.  , -4.5 , -9.  ],
       [ 1.5 ,  0.25,  1.  , -5.5 , -6.  ],
       [ 1.  ,  1.  , -1.  ,  0.  ,  0.  ],
       [ 4.  ,  0.25, -1.  , -0.5 ,  0.  ],
       [-1.  ,  1.  , -2.5 ,  0.  ,  0.  ],
       [-2.  , -1.  ,  0.5 ,  5.5 ,  2.  ]], dtype=float32)

In [27]:
# Save the model summary to a text file (Keras 3 style)
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

# Convert and synthesize with HLS4ML
Uses KV260 (xck26-sfvc784-2LV-c). You need to set the xpfm-path manually (it should be set based on env-path in source code?)

In [28]:
import hls4ml
hls_model = hls4ml.converters.convert_from_keras_model(
    model, 
    backend='Vitis', 
    project_name=f'{model_to_test}_{model_revision}_hls4ml_prj_{hls4ml_revision}',
    output_dir=output_dir, 
    part='xck26-sfvc784-2LV-c',
    strategy = 'latency',
    reusefactor = 4,
    #io_type='io_stream' # Must be parallel
)
hls_model.compile()

Test the HLS-model if it actually works

In [29]:
hls_model.predict(np.ascontiguousarray(x_test[:10]))

array([[ 2.5 ,  1.  , -1.5 ,  0.  ,  0.  ],
       [-1.  ,  1.75, -3.5 ,  0.  ,  0.  ],
       [-0.5 , -0.25,  1.  , -2.5 , -5.  ],
       [-1.  , -1.  ,  4.  , -3.  , -6.  ],
       [ 0.  , -0.75,  4.  , -4.5 , -9.  ],
       [ 1.5 ,  0.25,  1.  , -5.5 , -6.  ],
       [ 1.  ,  1.  , -1.  ,  0.  ,  0.  ],
       [ 4.  ,  0.25, -1.  , -0.5 ,  0.  ],
       [-1.  ,  1.  , -2.5 ,  0.  ,  0.  ],
       [-2.  , -1.  ,  0.5 ,  5.5 ,  2.  ]], dtype=float32)

In [30]:
hls_model.build(
    csim=False,
    #synth=True, 
    #bitfile=True
    ) 


****** vitis-run v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [vitis-run 82-31] Launching vitis_hls: vitis_hls -nolog -run tcl -f /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_latency_reusefactor4/build_prj.tcl -work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_latency_reusefactor4

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2023.2 (64-bit)
  **** SW Build 4023990 on Oct 11 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis_HLS/2023.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] Ru

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.573',
  'BestLatency': '8',
  'WorstLatency': '8',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'DSP': '5',
  'FF': '2853',
  'LUT': '13271',
  'BRAM_18K': '0',
  'URAM': '0',
  'AvailableBRAM_18K': '288',
  'AvailableDSP': '1248',
  'AvailableFF': '234240',
  'AvailableLUT': '117120',
  'AvailableURAM': '64'}}

In [ ]:
hls4ml.report.read_vivado_report(os.path.join(output_dir))

Found 1 solution(s) in /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_latency_reusefactor4/hgq2_1_hls4ml_prj_Vitis_latency_reusefactor4_prj.
Reports for solution "solution1":

C simulation report not found.
SYNTHESIS REPORT:
== Vitis HLS Report for 'hgq2_1_hls4ml_prj_Vitis_latency_reusefactor4'
* Date:           Wed Mar 18 11:54:48 2026

* Version:        2023.2 (Build 4023990 on Oct 11 2023)
* Project:        hgq2_1_hls4ml_prj_Vitis_latency_reusefactor4_prj
* Solution:       solution1 (Vivado IP Flow Target)
* Product family: zynquplus
* Target device:  xck26-sfvc784-2LV-c


== Performance Estimates
+ Timing: 
    * Summary: 
    +--------+---------+----------+------------+
    |  Clock |  Target | Estimated| Uncertainty|
    +--------+---------+----------+------------+
    |ap_clk  |  5.00 ns|  3.573 ns|     1.35 ns|
    +--------+---------+----------+------------+

+ Latency: 
    * Summary: 
    +---------+---------+-----------+-----------+-----+